## tl;dr

当前 v0.3.0 的两个语向 COMET 已复测，均高于本次 Google ML Kit 端侧结果。性能场次只完成 10/24 个进程，不能标为完整基线。此笔记本只复核已归档数据，不连接手机，也不重新运行大模型评分。

## Context & Methods

### Key Assumptions

- 固定 FLORES-200 devtest 前 200 条，COMET `wmt22-comet-da × 100`；只比较本批语料。
- 相同引擎/语向的重复译文复用分数，不增加独立语料数量。
- 新版与 ML Kit 的质量对比，不等于旧版→新版质量不变。
- 性能完整性与质量输出完整性分开验证；app PSS 不与 native RSS 混算。
- 原始记录、失败原因和重跑方法见同目录 README；依赖版本、checkpoint/参考哈希见评分 JSON。

### 1. Setup

In [1]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
from IPython.display import Markdown, display

root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / 'tools/app-bench/run.py').is_file())
record = root / 'benchmarks/v0.3.0/mi14-2026-09-09'
def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


## Data

### 2. Verify source lineage

In [2]:
quality_path = record / 'quality-comet-completed.json'
app_dir = record / 'app-measure-foreground'
quality = json.loads(quality_path.read_text())
app = json.loads((app_dir / 'results.json').read_text())
assert quality['status'] == 'complete'
assert quality['archived_source_report_sha256'] == digest(app_dir / 'results.json')
assert app['source_files_sha256'] == digest(app_dir / 'source-files.json')
for run in quality['runs']:
    archived_file = app_dir / Path(run['source_file']).name
    assert run['source_sha256'] == digest(archived_file)
print('Source report, source manifest, and all scored raw-file hashes verified.')


Source report, source manifest, and all scored raw-file hashes verified.


### 3. Check measurement coverage

In [3]:
completed = [r for r in app['runs'] if r.get('status') == 'complete']
keys = {(r['direction'], r['engine'], r['threads']) for r in completed}
assert len(completed) == quality['completed_processes'] == 10
assert len(keys) == 8
assert quality['planned_processes'] == 24
assert len(quality['uncompleted_cells']) == 14
assert len(quality['scored_outputs']) == 4
assert quality['performance_validated'] is False
print({'complete_processes': len(completed), 'planned_processes': 24,
       'covered_scenarios': len(keys), 'unique_translation_sets': 4})


{'complete_processes': 10, 'planned_processes': 24, 'covered_scenarios': 8, 'unique_translation_sets': 4}


## Results

### 4. Rebuild the quality comparison

In [4]:
scores = {}
for run in quality['runs']:
    engine = run['label'].split('-')[0]
    key = (engine, run['direction'])
    if key in scores:
        assert abs(scores[key] - run['comet_x100']) < 1e-10
    scores[key] = run['comet_x100']
rows = ['| Direction | ML Kit | v0.3.0 | Difference |', '|---|---:|---:|---:|']
for direction in ['enzh', 'jazh']:
    before, after = scores[('mlkit', direction)], scores[('bergamot', direction)]
    rows.append(f'| {direction} | {before:.2f} | {after:.2f} | {after-before:+.2f} |')
display(Markdown('\n'.join(rows)))


| Direction | ML Kit | v0.3.0 | Difference |
|---|---:|---:|---:|
| enzh | 72.69 | 87.27 | +14.57 |
| jazh | 68.93 | 86.71 | +17.79 |

### 5. Confirm that incomplete performance evidence is rejected

In [5]:
app_check = subprocess.run([sys.executable, str(root / 'tools/app-bench/check.py'), str(app_dir)],
                           capture_output=True, text=True)
app_verdict = json.loads(app_check.stdout)
assert app_check.returncode == 2 and app_verdict['accepted'] is False
print('App evidence:', app_verdict['errors'])
native_path = record / 'native-mi14-attempt1.json'
native_check = subprocess.run([sys.executable, str(root / 'tools/version-bench/check.py'),
    str(native_path), str(native_path), '--baseline-version', 'v0.2.0', '--candidate-version', 'v0.3.0'],
    capture_output=True, text=True)
assert native_check.returncode == 2
print('Native evidence:', (native_check.stdout + native_check.stderr).strip())


App evidence: ['requires a completed measure run, not prepare/quality/partial data', 'post-run model fingerprints missing; model stability not verified', 'expected 24 unique runs: all eight scenarios x three rounds', 'jazh/bergamot/1t: incomplete process']
Native evidence: INVALID: v0.2.0: missing or unexpected scenarios


## Takeaways

新版在本次两个语向的 COMET 高于 ML Kit，已导出的不同线程档位译文一致。三轮性能、采集结束时模型指纹和旧版质量对照仍缺失；重新连接固定设备后另开完整场次，不能把中断片段拼接成通过结果。